# Module 05: Scikit-Learn for Machine Learning
## Notebook 03: Supervised Classification Models and Imbalanced Datasets

Classification predicts discrete categorical outcomes (e.g. churn vs. retained, fraudulent vs. legitimate, disease diagnosis). This notebook covers linear classifiers, tree ensembles, class imbalance handling, threshold tuning, and probability calibration.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Train and interpret **Logistic Regression** and **Random Forest Classifiers**.
2. Address class imbalance using **`class_weight='balanced'`**.
3. Evaluate models beyond raw accuracy: **Confusion Matrices, Precision, Recall, and F1-Score**.
4. Plot and interpret **ROC Curves and Area Under the Curve (ROC-AUC)**.
5. **Advanced:** Calibrate model confidence scores using **`CalibratedClassifierCV`** and validate with **Reliability Curves**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, roc_auc_score
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

print("Scikit-Learn loaded!")

### 1. Generating an Imbalanced Binary Classification Problem

In fraud detection or disease screening, positive events represent a tiny fraction (< 10%) of all samples:

In [ ]:
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    weights=[0.90, 0.10],  # 90% Class 0 (Negative), 10% Class 1 (Positive)
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Total Test Samples: {len(y_test)}")
print(f"Class Distribution: {np.bincount(y_test)} (Only {np.bincount(y_test)[1]} positives!)")

---
### 2. Training and Comparing Classifiers

We train:
1. **Standard Logistic Regression** (Unweighted).
2. **Cost-Sensitive Logistic Regression** (`class_weight='balanced'`).
3. **Random Forest Classifier** (`n_estimators=100`).

In [ ]:
# 1. Unweighted Logistic Regression
lr_unweighted = LogisticRegression(random_state=42).fit(X_train, y_train)

# 2. Balanced Logistic Regression (Penalizes minority class errors inversely proportional to class frequency)
lr_balanced = LogisticRegression(class_weight='balanced', random_state=42).fit(X_train, y_train)

# 3. Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42).fit(X_train, y_train)

print("=== Balanced Logistic Regression Report ===")
print(classification_report(y_test, lr_balanced.predict(X_test), target_names=['Majority (0)', 'Minority (1)']))

print("\n=== Random Forest Report ===")
print(classification_report(y_test, rf.predict(X_test), target_names=['Majority (0)', 'Minority (1)']))

---
### 3. Confusion Matrix Visualization

A Confusion Matrix details:
- **True Negatives (TN)** & **False Positives (FP)**
- **False Negatives (FN)** (Critically dangerous in disease diagnosis / fraud)
- **True Positives (TP)**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

ConfusionMatrixDisplay.from_estimator(lr_balanced, X_test, y_test, cmap='Blues', ax=ax1)
ax1.set_title("Logistic Regression (Balanced)", fontweight='bold')

ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, cmap='Greens', ax=ax2)
ax2.set_title("Random Forest (Balanced)", fontweight='bold')

plt.tight_layout()
plt.show()

---
### 4. ROC Curves and Area Under the Curve (AUC)

The ROC curve measures discriminatory power across all possible probability classification thresholds:

In [ ]:
# Extract predicted probabilities for minority class
y_prob_lr = lr_balanced.predict_proba(X_test)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_rf = roc_auc_score(y_test, y_prob_rf)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Reg (AUC = {auc_lr:.3f})', linewidth=2)
ax.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc_rf:.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random Chance (AUC = 0.500)')

ax.set_title("ROC-AUC Comparison on Imbalanced Dataset", fontsize=12, fontweight='bold')
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.legend(loc='lower right')
ax.grid(True, linestyle=':', alpha=0.5)

plt.show()

---
### 5. Advanced Complex Usage: Probability Calibration (`CalibratedClassifierCV`) & Reliability Curves

In high-stakes applications (medical risk, credit underwriting):
- Predicted probability should equal true empirical probability: If a model predicts $P(\text{Default}) = 0.80$ for 100 people, exactly 80 should default.
- Many complex models (Random Forests, SVMs, Boosted Trees) output **uncalibrated scores**: they tend to push probabilities away from 0 and 1 or produce distorted distributions.
- **`CalibratedClassifierCV`**: Fits a calibrator (Platt Scaling via `'sigmoid'` or non-parametric `'isotonic'` regression) via internal cross-validation.

In [ ]:
# Calibrate Random Forest using Sigmoid (Platt Scaling)
calibrated_rf = CalibratedClassifierCV(estimator=rf, method='sigmoid', cv=3)
calibrated_rf.fit(X_train, y_train)

# Calculate calibration curves (Reliability Diagrams)
prob_uncal = rf.predict_proba(X_test)[:, 1]
prob_cal = calibrated_rf.predict_proba(X_test)[:, 1]

fraction_of_positives_uncal, mean_predicted_uncal = calibration_curve(y_test, prob_uncal, n_bins=8)
fraction_of_positives_cal, mean_predicted_cal = calibration_curve(y_test, prob_cal, n_bins=8)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([0, 1], [0, 1], 'k:', label='Perfectly Calibrated (Ideal)')
ax.plot(mean_predicted_uncal, fraction_of_positives_uncal, 's--', color='crimson', label='Uncalibrated Random Forest')
ax.plot(mean_predicted_cal, fraction_of_positives_cal, 'o-', color='teal', linewidth=2, label='Calibrated Random Forest (Platt)')

ax.set_title("Reliability Diagram: Model Probability Calibration", fontsize=12, fontweight='bold')
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Empirical Fraction of Positives")
ax.legend(loc='upper left')
ax.grid(True, linestyle=':', alpha=0.5)

plt.show()
print(f"Uncalibrated Brier Score Loss: {np.mean((prob_uncal - y_test)**2):.4f}")
print(f"Calibrated Brier Score Loss:   {np.mean((prob_cal - y_test)**2):.4f} (Lower is better!)")

### Summary & Next Steps
In this notebook, you mastered:
- Binary classification algorithms and decision boundaries.
- Addressing class imbalance with cost-sensitive `class_weight='balanced'`.
- Evaluating models using Confusion Matrices, Precision, Recall, and F1.
- ROC curves and discriminatory ranking with ROC-AUC.
- Probability calibration (`CalibratedClassifierCV`) and Reliability Diagram evaluation.

**Next Notebook:** `04_unsupervised_clustering_and_pca.ipynb` — PCA dimensionality reduction, scree plots, K-Means, elbow/silhouette analysis, and DBSCAN density clustering.